# 14주차: 실패를 다시 볼 수 있는 운영 기록

- Python 표준 라이브러리, CPU, 설치·학습·GPU 불필요
- 제주 공개 문서 고정판의 행정시 사실과 교수 작성 질문 사용
- 규칙형 QA 운영 모형. 실제 언어모델의 능력·운영 품질 실측 아님
- 제공된 로거 연결 함수 한 곳만 교체. 기본 그대로 전체 실행 가능
- 실행 폴더에 `week14-evidence/` 생성. 재실행 시 이번 실습 파일만 갱신
- 원문 출처·저자: https://ko.wikipedia.org/w/index.php?oldid=42464087 (CC BY-SA 4.0)


In [ ]:
import json, hashlib, html, re, time, urllib.parse, urllib.request
from pathlib import Path
from collections import Counter
OLDID = 42464087
SOURCE = f"https://ko.wikipedia.org/w/index.php?oldid={OLDID}"
print("공개 문서·저자 목록:", SOURCE, "CC BY-SA 4.0")
source_verified = False
api = "https://ko.wikipedia.org/w/api.php?" + urllib.parse.urlencode({"action":"parse", "oldid":OLDID, "prop":"text", "format":"json", "formatversion":2})
try:
    request = urllib.request.Request(api, headers={"User-Agent":"deepnlp-course/1.0 (yj.lee@chu.ac.kr)"})
    with urllib.request.urlopen(request, timeout=20) as response:
        payload = json.load(response)
    assert payload["parse"]["revid"] == OLDID, "고정판 번호 불일치"
    assert payload["parse"]["title"] == "제주특별자치도", "원문 제목 불일치"
    article = html.unescape(re.sub(r"<[^>]+>", " ", payload["parse"]["text"]))
    source_verified = "제주시" in article and "서귀포시" in article
    print("고정판 행정시 이름 확인:", source_verified)
except Exception as error:
    print("공개 사실 확인: 미확인", type(error).__name__)
print("아래 fixture는 작성 예시. 출처 조회 실패를 사실 확인 성공으로 대체하지 않음")


## 1. 제공된 프로토타입과 버전 묶음

- 공개 사실에 대응하는 작성 레코드: 두 행정시 이름
- v1: `행정시` 포함 질문에서 이름과 수 반환
- prompt-v2: 이름만 반환하는 출력 정책 변경
- model-v2: count에 1을 더하는 오류를 의도적으로 넣은 규칙 정책 변경
- 이 모델 버전은 학습된 가중치 버전이 아닌 작성 규칙 버전
- 새 질문의 정답 기준이 있는데도 답을 못 하면 미응답 실패. 라벨이 없으면 정확도 미측정


In [ ]:
def digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

PIPELINE_SOURCE = """def pipeline(query, config, retrieval):
    if "행정시" not in query:
        return {"text":"판단 보류", "count":None, "error":"unsupported_input"}
    names = retrieval["names"]
    count = len(names) + (1 if config["model_version"] == "rule-v2" else 0)
    text = ", ".join(names)
    if config["prompt_version"] == "prompt-v1":
        text += f", {count}곳"
    return {"text":text, "count":count if config["prompt_version"] == "prompt-v1" else None, "error":None}
"""
exec(PIPELINE_SOURCE)
RETRIEVAL = {"version":"jeju-authored-v1", "source_oldid":OLDID, "names":["제주시", "서귀포시"]}
CONFIGS = {
    "baseline": {"model_version":"rule-v1", "prompt_version":"prompt-v1"},
    "prompt-change": {"model_version":"rule-v1", "prompt_version":"prompt-v2"},
    "model-change": {"model_version":"rule-v2", "prompt_version":"prompt-v1"},
}
FIXTURES = {
    "F1": {"query":"제주 두 도시의 총 개수는?", "query_version":"authored-q-v1", "config_id":"baseline", "expected_count":2, "path":"input_shift"},
    "F2": {"query":"제주의 행정시는 몇 곳?", "query_version":"authored-q-v1", "config_id":"prompt-change", "expected_count":2, "path":"prompt_revision"},
    "F3": {"query":"제주의 행정시는 몇 곳?", "query_version":"authored-q-v1", "config_id":"model-change", "expected_count":2, "path":"model_revision"},
}
BUNDLE = {"pipeline_source":PIPELINE_SOURCE, "retrieval":RETRIEVAL, "configs":CONFIGS, "fixtures":FIXTURES, "source":SOURCE}
CODE_HASH = digest(PIPELINE_SOURCE)
BUNDLE_HASH = digest(BUNDLE)
baseline = pipeline("제주의 행정시는 몇 곳?", CONFIGS["baseline"], RETRIEVAL)
assert baseline["count"] == 2
print("기준 실행:", baseline)
print("코드 hash:", CODE_HASH, "묶음 hash:", BUNDLE_HASH)


## 2. 제공 로거와 모니터링

- 기본: 입력·출력 원문 없이 ID와 hash, 버전, 오류·판정·지연 저장
- 대안: 이번 공개 작성 질의만 함께 저장. 실제 사용자 원문에는 적용 금지
- hash는 익명화 보장이 아님. 짧은 질의는 후보 대입으로 복원 가능
- 재현에 필요한 작성 질의는 별도 공개 fixture registry에 존재
- 운영 개인질의를 같은 방식으로 registry에 복사하지 않는 원칙


In [ ]:
def minimal_logger(event, fixture, answer):
    return dict(event)

def public_fixture_logger(event, fixture, answer):
    record = dict(event)
    record["public_authored_query"] = fixture["query"]
    return record

def attach_monitor(logger):
    def run(fixture_id):
        fixture = FIXTURES[fixture_id]
        config = CONFIGS[fixture["config_id"]]
        start = time.perf_counter()
        answer = pipeline(fixture["query"], config, RETRIEVAL)
        elapsed_ms = (time.perf_counter() - start) * 1000
        label = fixture.get("expected_count")
        correct = None if label is None else answer["count"] == label
        stale_pass = all(name in answer["text"] for name in RETRIEVAL["names"])
        event = {
            "id":fixture_id, "query_version":fixture["query_version"],
            "query_hash":digest(fixture["query"]), "output_hash":digest(answer),
            "config_id":fixture["config_id"], "config_hash":digest(config),
            **config, "retrieval_version":RETRIEVAL["version"], "retrieval_hash":digest(RETRIEVAL),
            "code_hash":CODE_HASH, "bundle_hash":BUNDLE_HASH, "source_oldid":OLDID,
            "source_verified":source_verified, "latency_ms":elapsed_ms,
            "error":answer["error"], "correct":correct, "outcome":"unknown" if correct is None else "pass" if correct else "fail",
            "metric_version":"exact-count-v2", "stale_metric_version":"names-present-v1", "stale_pass":stale_pass,
        }
        return logger(event, fixture, answer)
    return run


## 3. 한 곳 연결

- 기본 실행 후 아래 `minimal_logger`만 `public_fixture_logger`로 교체하고 전체 실행
- 두 함수 모두 이미 제공됨. 새 로거·재현 알고리즘 구현 불필요
- 정책 차이: 저장 열에 공개 작성 질의 원문을 추가하는지 여부


In [ ]:
# TODO: minimal_logger를 제공된 public_fixture_logger로 교체
run_with_monitor = attach_monitor(minimal_logger)


## 4. 실제 실행에서 실패 3건 수집

- 출력 정확도와 이전 지표 통과율을 따로 관찰
- 지연은 현재 CPU 실행의 실측. 외부 서비스의 지연·비용으로 일반화 불가
- 파일 저장 후 읽어 다시 검증


In [ ]:
records = [run_with_monitor(case_id) for case_id in FIXTURES]
assert len(records) == 3 and all(record["correct"] is False for record in records)
folder = Path("week14-evidence")
folder.mkdir(exist_ok=True)
(folder / "bundle.json").write_text(json.dumps(BUNDLE, ensure_ascii=False, indent=2), encoding="utf-8")
log_path = folder / "failures.jsonl"
log_path.write_text("".join(json.dumps(record, ensure_ascii=False) + "\n" for record in records), encoding="utf-8")
print("ID | 원인 경로 | 새 지표 | 이전 지표 | 오류 | CPU 지연(ms)")
for record in records:
    print(record["id"], FIXTURES[record["id"]]["path"], record["outcome"], record["stale_pass"], record["error"], round(record["latency_ms"], 4))
known = [record for record in records if record["correct"] is not None]
print("라벨 확인:", len(known), "/", len(records))
print("exact-count-v2:", sum(record["correct"] for record in known), "/", len(known))
print("names-present-v1:", sum(record["stale_pass"] for record in records), "/", len(records))
print("오류 유형:", dict(Counter(record["error"] or "no_exception" for record in records)))
print("라벨 미수집 예시: 정확도 unknown, 0으로 대체 금지")


## 5. 저장된 묶음으로 재현

- 저장된 공개 fixture·config·코드를 로드. 제공된 코드 문자열 이외의 파일로 대체하면 검증 실패
- 로그 원문에 기대 답을 더 넣어 오류를 감추지 않고 registry와 hash 대조
- 재현 대상: 의미 출력·오류·판정. 현재 측정한 지연의 byte 일치 대상 아님
- 임의 외부 코드 파일은 `exec` 대상에 넣지 않음


In [ ]:
saved = json.loads((folder / "bundle.json").read_text(encoding="utf-8"))
assert saved["pipeline_source"] == PIPELINE_SOURCE, "제공 코드 이외의 실행 금지"
assert digest(saved) == BUNDLE_HASH
namespace = {}
exec(saved["pipeline_source"], namespace)
loaded = [json.loads(line) for line in log_path.read_text(encoding="utf-8").splitlines()]
for record in loaded:
    fixture = saved["fixtures"][record["id"]]
    config = saved["configs"][record["config_id"]]
    assert config["model_version"] == record["model_version"]
    assert config["prompt_version"] == record["prompt_version"]
    assert fixture["query_version"] == record["query_version"]
    assert saved["retrieval"]["version"] == record["retrieval_version"]
    assert saved["retrieval"]["source_oldid"] == record["source_oldid"]
    assert record["metric_version"] == "exact-count-v2"
    assert record["stale_metric_version"] == "names-present-v1"
    assert digest(config) == record["config_hash"]
    assert digest(saved["retrieval"]) == record["retrieval_hash"]
    assert digest(saved["pipeline_source"]) == record["code_hash"]
    assert digest(saved) == record["bundle_hash"]
    assert digest(fixture["query"]) == record["query_hash"]
    answer = namespace["pipeline"](fixture["query"], config, saved["retrieval"])
    assert digest(answer) == record["output_hash"]
    assert answer["error"] == record["error"]
    assert (answer["count"] == fixture["expected_count"]) == record["correct"]
    assert all(name in answer["text"] for name in saved["retrieval"]["names"]) == record["stale_pass"]
    assert record["outcome"] == "fail" and record["latency_ms"] >= 0
    print("재현 일치:", record["id"], answer)
print("실패 3건 재현 완료. 저장 위치:", folder.resolve())


## 6. 수신자 기준과 멘토 리뷰 자료

| 사례 | 실행 관찰 | 이용자 영향 | 개발·운영·소통 중 우선 | 제시 근거 | 멘토 질문 | 상정 조치 |
|---|---|---|---|---|---|---|
| F1 | 새 표현에서 판단 보류 | 공개 사실 문의에도 응답 없음 | 소통·운영 | F1 로그·묶음 | 보류와 재문의 안내가 충분한지 | 표현 확장 전 보류 안내 |
| F2 | 이름만 출력, count 없음 | 사람은 읽지만 수신 프로그램은 필드 누락 | 개발 | F2·prompt-v2 | 누락 필드가 자동 집계에 미치는 영향 | 버전 비교 후 prompt-v1 복귀 |
| F3 | 두 이름인데 3곳 | 잘못된 수치 전달 | 운영·소통 | F3·rule-v2 | 오답 통지와 중단 중 우선 | 해당 버전 자동 출력 보류 |

- 위 표는 교수 작성 검토안. 실제 멘토의 의견·승인·수정 지시 아님
- 실제 리뷰에서 별도 기록할 열: 멘토 판단, 변경 우선순위, 담당 역할, 재검증 근거, 미확인 조건
- Q2 수신자 구분: 읽는 사람은 출처·오답 영향, 프로그램은 필드·schema, 운영자는 버전·재현성을 확인

## 돌아보기

1. F1과 F2 중 입력이 바뀐 사례와 프롬프트가 바뀐 사례 구분
2. 새 지표 0/3과 이전 지표 2/3의 차이를 실패 출력으로 설명
3. 코드·질의·config hash만 있고 공개 fixture registry가 없을 때 재현 가능한지 설명
4. 수정 전·후 로깅 열과 원문 최소화의 차이 기록
5. 실제 멘토 의견은 미수신이면 미수신으로 표시. 이름·연락처·API 키·실제 사용자 원문 제외


## 제출 안내

- 별도 제출 공지가 있을 때 `week14-evidence/bundle.json`, `failures.jsonl`과 실패 3건의 관찰·멘토 질문 표를 함께 제출
- 묶음은 제공된 공개 작성 fixture·config·코드만 포함, 로그는 선택한 최소 기록 정책과 연결
- 실제 멘토의 의견과 교수 작성 검토안을 구분. 미수신·미확인은 그대로 표시
- 학번 폴더만 사용, 이름·연락처·API 키·실제 사용자 원문 제외
